# 🔥 Mytho — Recurrent-Depth Transformer: T4 Pretraining**Architecture highlights:**- Recurrent depth with Adaptive Computation Time (ACT)- Multi-Latent Attention (DeepSeek-V2 inspired KV compression)- Mixture-of-Experts with SwiGLU + load-balancing- Latent scratchpad for internal reasoning- Verifier-guided halting**T4 optimisations (16 GB VRAM):**- FP16 mixed-precision via AMP (T4 lacks BF16)- `max_depth=8` (reduced from 12)- `seq_len=512` (reduced from 2048)- Gradient accumulation (effective batch = 32)- Optional gradient checkpointing> ⚙️ **Runtime → Change runtime type → T4 GPU** before running.

## 1 · Environment Setup

In [ ]:
# ── Verify GPU ───────────────────────────────────────────────────
import torch
assert torch.cuda.is_available(), "❌ No GPU — go to Runtime → Change runtime type → T4 GPU"
gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"✅ GPU: {gpu}  |  VRAM: {vram:.1f} GB")

In [ ]:
# ── Install dependencies ─────────────────────────────────────────
!pip install -q torch>=2.1 tiktoken>=0.5 datasets>=2.16 tqdm>=4.65 matplotlib
print("✅ Dependencies installed")

## 2 · Clone RepositoryPulls the latest Mytho source from GitHub.

In [ ]:
# ── Clone Mytho from GitHub ──────────────────────────────────────
import os

REPO_URL = "https://github.com/dev-760/Mytho.git"
REPO_DIR = "Mytho"

if not os.path.exists(REPO_DIR):
    !git clone $REPO_URL $REPO_DIR
else:
    print(f"'{REPO_DIR}' already exists — pulling latest...")
    %cd $REPO_DIR
    !git pull
    %cd ..

# Change working directory into the repo
%cd $REPO_DIR
print(f"\n✅ Working directory: {os.getcwd()}")
!ls mytho_model/

## 3 · Build & Verify Model

In [ ]:
# ── Build model with T4-optimised config ─────────────────────────
from mytho_model import MythoConfig, MythoModel
from data import VOCAB_SIZE

config = MythoConfig(
    vocab_size=VOCAB_SIZE,   # 50257 (GPT-2 BPE)
    d_model=768,
    n_heads=12,
    d_head=64,
    d_latent_kv=256,
    d_rope=32,
    n_experts=8,
    n_active_experts=2,
    d_expert_ff=2048,
    max_depth=8,             # ← reduced for T4
    max_seq_len=512,         # ← reduced for T4
    dropout=0.0,
)

model = MythoModel(config).cuda()
n_params = model.num_parameters()
print(f"✅ Model built: {n_params:,} parameters ({n_params/1e6:.1f}M)")
print(f"   Vocab: {config.vocab_size}  |  d_model: {config.d_model}")
print(f"   Heads: {config.n_heads}  |  Experts: {config.n_experts} (top-{config.n_active_experts})")
print(f"   Max depth: {config.max_depth}  |  Seq len: {config.max_seq_len}")

# Quick forward pass test
ids = torch.randint(1, config.vocab_size, (2, 64), device="cuda")
with torch.no_grad():
    out = model(ids, labels=ids)
print(f"\n   Forward pass OK — Loss: {out['loss'].item():.4f}, Depth: {out['mean_depth'].item():.1f}")

del model, ids, out
torch.cuda.empty_cache()

## 4 · Data PipelineStreams [FineWeb-Edu](https://huggingface.co/datasets/HuggingFaceFW/fineweb-edu) and tokenises with GPT-2 BPE (tiktoken).

In [ ]:
# ── Test tokeniser + data stream ─────────────────────────────────
from data import tokenise, decode, create_dataloader, VOCAB_SIZE

# Tokeniser roundtrip
text = "The recurrent depth transformer processes tokens iteratively."
ids = tokenise(text)
assert decode(ids) == text
print(f"✅ Tokeniser OK — {len(ids)} tokens for {len(text)} chars")

# Test dataloader (small batch)
print("\n⏳ Testing data stream (this downloads a small sample)...")
dl = create_dataloader(seq_len=128, batch_size=2, max_docs=10, num_workers=0)
batch_x, batch_y = next(iter(dl))
print(f"✅ Data OK — batch shape: {batch_x.shape}")
print(f"   Sample tokens: {batch_x[0, :20].tolist()}")
print(f"   Decoded: {decode(batch_x[0, :40].tolist())[:80]}...")
del dl, batch_x, batch_y

## 5 · Google Drive (Optional)Mount Drive to save checkpoints persistently.

In [ ]:
# ── Mount Google Drive (uncomment to enable) ────────────────────
# from google.colab import drive
# drive.mount('/content/drive')
# CKPT_DIR = "/content/drive/MyDrive/mytho_checkpoints"

# ── Or use local storage ────────────────────────────────────────
CKPT_DIR = "checkpoints_t4"
print(f"✅ Checkpoints will save to: {CKPT_DIR}")

## 6 · Pretraining**T4 VRAM budget (~16 GB):**| Component | Estimate ||---|---|| Model params (FP16) | ~340 MB || Optimizer states (FP32) | ~1.4 GB || Gradients (FP16) | ~340 MB || Activations (batch=4, depth=8) | ~2-4 GB || **Total** | **~5-6 GB** ✅ |Adjust `--max_steps` for longer runs. `500` steps ≈ 5 min, `5000` steps ≈ 45 min on T4.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  🏋️  RUN PRETRAINING
# ═══════════════════════════════════════════════════════════════════

# Quick smoke test (≈2 min) — change max_steps for longer training
!python pretrain_t4.py \
    --max_steps 500 \
    --batch_size 4 \
    --grad_accum 8 \
    --seq_len 512 \
    --max_depth 8 \
    --lr 3e-4 \
    --warmup_steps 50 \
    --log_every 10 \
    --save_every 250 \
    --ckpt_dir {CKPT_DIR} \
    --max_docs 5000

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  🏋️  FULL PRETRAINING (uncomment to run — ≈45 min for 5k steps)
# ═══════════════════════════════════════════════════════════════════

# !python pretrain_t4.py \
#     --max_steps 5000 \
#     --batch_size 4 \
#     --grad_accum 8 \
#     --seq_len 512 \
#     --max_depth 8 \
#     --lr 3e-4 \
#     --warmup_steps 200 \
#     --log_every 10 \
#     --save_every 500 \
#     --ckpt_dir {CKPT_DIR}

## 7 · Training Metrics

In [ ]:
# ── Plot training curves ─────────────────────────────────────────
import json, matplotlib.pyplot as plt
from pathlib import Path

log_path = Path(CKPT_DIR) / "train_log.jsonl"
if log_path.exists():
    records = [json.loads(l) for l in log_path.read_text().strip().split("\n") if l.strip()]
    steps  = [r["step"] for r in records]
    losses = [r["loss"] for r in records]
    ce     = [r["ce_loss"] for r in records]
    depths = [r["mean_depth"] for r in records]
    lrs    = [r["lr"] for r in records]

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    fig.suptitle("Mytho T4 Pretraining", fontsize=16, fontweight="bold")

    axes[0,0].plot(steps, losses, color="#e74c3c", linewidth=1.5)
    axes[0,0].set_title("Total Loss"); axes[0,0].set_xlabel("Step"); axes[0,0].grid(alpha=0.3)

    axes[0,1].plot(steps, ce, color="#3498db", linewidth=1.5)
    axes[0,1].set_title("Cross-Entropy Loss"); axes[0,1].set_xlabel("Step"); axes[0,1].grid(alpha=0.3)

    axes[1,0].plot(steps, depths, color="#2ecc71", linewidth=1.5)
    axes[1,0].set_title("Mean ACT Depth"); axes[1,0].set_xlabel("Step"); axes[1,0].grid(alpha=0.3)

    axes[1,1].plot(steps, lrs, color="#9b59b6", linewidth=1.5)
    axes[1,1].set_title("Learning Rate"); axes[1,1].set_xlabel("Step"); axes[1,1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(CKPT_DIR) / "training_curves.png", dpi=150)
    plt.show()
    print(f"\n✅ Final loss: {losses[-1]:.4f}  |  Final depth: {depths[-1]:.1f}")
else:
    print("⚠ No training log found. Run training first.")

## 8 · Text Generation

In [ ]:
# ── Generate text from checkpoint ────────────────────────────────
import torch, glob
from pathlib import Path
from mytho_model import MythoModel
from data import tokenise, decode

# Find latest checkpoint
ckpts = sorted(glob.glob(f"{CKPT_DIR}/step_*.pt"))
if ckpts:
    ckpt_path = ckpts[-1]
    print(f"▸ Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location="cuda", weights_only=False)
    config = ckpt["config"]
    model = MythoModel(config).cuda()
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"▸ Loaded at step {ckpt['step']}  |  {model.num_parameters():,} params")

    # Generate from several prompts
    prompts = [
        "The history of artificial intelligence",
        "In mathematics, a prime number is",
        "The scientific method involves",
    ]

    for prompt in prompts:
        ids = tokenise(prompt)
        input_ids = torch.tensor([ids], device="cuda")

        with torch.no_grad():
            output = model.generate(
                input_ids, max_new_tokens=100,
                temperature=0.8, top_k=50, top_p=0.9,
            )

        generated = decode(output[0].tolist())
        print(f"\n{'─'*60}")
        print(f"Prompt: {prompt}")
        print(f"Output: {generated}")

    del model
    torch.cuda.empty_cache()
else:
    print("⚠ No checkpoints found. Run training first.")

## 9 · Advanced Features DemoTest the scratchpad, verifier, branching, and expert monitoring.

In [ ]:
# ── Advanced feature demos ───────────────────────────────────────
import torch
from mytho_model import (
    MythoConfig, MythoModel,
    LatentScratchpad, VerifierHead, BranchingController,
    ExpertMetrics, DynamicExpertGrowth,
)

cfg = MythoConfig(
    d_model=256, n_heads=4, d_head=64, d_latent_kv=64,
    d_rope=16, max_depth=4, n_experts=4, n_active_experts=2,
    d_expert_ff=512, max_seq_len=64, vocab_size=1000,
)
ids = torch.randint(1, 1000, (2, 32), device="cuda")

# 1. Base model
m = MythoModel(cfg).cuda()
out = m(ids, labels=ids)
print(f"[Base]        Loss: {out['loss'].item():.4f}  Depth: {out['mean_depth'].item():.1f}")

# 2. Scratchpad + Verifier-Driven ACT
ms = MythoModel(cfg, use_scratchpad=True, d_scratch=64).cuda()
out_s = ms(ids, labels=ids)
print(f"[Scratchpad]  Loss: {out_s['loss'].item():.4f}  "
      f"Conf: {out_s.get('confidence', torch.tensor(0)).item():.4f}")

# 3. Branching recurrence
mb = MythoModel(cfg, use_scratchpad=True, d_scratch=64,
                use_branching=True, n_branches=2).cuda()
out_b = mb(ids, labels=ids)
print(f"[Branching]   Loss: {out_b['loss'].item():.4f}  Depth: {out_b['mean_depth'].item():.1f}")

# 4. Expert metrics
em = ExpertMetrics(4)
logits = torch.randn(2, 32, 4)
topk_i = logits.topk(2, dim=-1).indices
em.update(logits, topk_i)
report = em.compute(m.blocks[0].moe)
print(f"[Experts]     Entropy: {report['expert_entropy']:.3f} / "
      f"{report['max_entropy']:.3f}  Stability: {report['routing_stability']:.3f}")

# 5. Generation
gen = ms.generate(ids[:1, :8], max_new_tokens=16)
print(f"[Generate]    Input: {ids[:1,:8].shape} → Output: {gen.shape}")

print("\n✅ All advanced features OK")
del m, ms, mb
torch.cuda.empty_cache()

## 10 · Scratchpad Pretraining (Optional)Enable the latent scratchpad + verifier-driven ACT for richer internal reasoning. Uses more VRAM (~2 GB extra).

In [ ]:
# ── Pretrain with scratchpad (optional — uses more VRAM) ─────────
# !python pretrain_t4.py \
#     --use_scratchpad \
#     --d_scratch 64 \
#     --max_steps 500 \
#     --batch_size 4 \
#     --grad_accum 8 \
#     --seq_len 512 \
#     --max_depth 8 \
#     --ckpt_dir checkpoints_t4_scratchpad \
#     --max_docs 5000